# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_10 — Statistical Arbitrage Pairs

### Research question

How can a pairs-trading statistical arbitrage strategy be researched responsibly using cointegration, spread modelling, train/test validation, and realistic transaction-cost diagnostics?

This notebook follows naturally from:

```text
03_05_vector_autoregression_granger
03_09_volatility_surface_alpha_signals
```

The goal is to build a disciplined pairs-trading research workflow.

It covers:

1. synthetic multi-asset universe generation;
2. correlation versus cointegration;
3. hedge-ratio estimation by OLS;
4. Engle-Granger-style pair screening;
5. spread stationarity diagnostics;
6. Ornstein-Uhlenbeck intuition;
7. half-life estimation;
8. rolling z-score spread signals;
9. train/test pair selection;
10. no-lookahead backtesting;
11. transaction-cost and turnover diagnostics;
12. false-pair demonstration;
13. parameter stability checks;
14. portfolio of selected pairs;
15. limitations and research extensions.

The key idea:

> Pairs trading is not “buy low, sell high” on two correlated assets. It is a bet that a hedged spread is mean-reverting, stable, and tradable after costs.

## 1. Correlation is not cointegration

Two assets can be highly correlated in returns but not form a stable spread.

Correlation asks whether short-term moves line up:

$$
corr(r^x_t,r^y_t)
$$

Cointegration asks whether a linear combination of price levels is stationary:

$$
s_t = y_t - \alpha - \beta x_t
$$

If $s_t$ is mean-reverting, then $x_t$ and $y_t$ may form a pairs-trading candidate.

A common trap:

> Two trending assets can have high price correlation even when their spread is not stable.

Pairs trading needs spread stability, not just correlation.

## 2. Hedge ratio

For log prices $x_t$ and $y_t$, estimate:

$$
y_t = \alpha + \beta x_t + \epsilon_t
$$

The spread is:

$$
s_t = y_t - \alpha - \beta x_t
$$

A market-neutral spread trade is:

- long $y$, short $\beta x$ when spread is low;
- short $y$, long $\beta x$ when spread is high.

The hedge ratio $\beta$ is estimated using training data only.

If $\beta$ is re-estimated using future data, that is look-ahead bias.

## 3. Spread z-score signal

A simple signal uses rolling z-score:

$$
z_t = \frac{s_t-\mu_{t,w}}{\sigma_{t,w}}
$$

Trading rule:

- if $z_t > entry$: short spread;
- if $z_t < -entry$: long spread;
- if $|z_t| < exit$: close position.

This rule assumes mean reversion.

If the spread is not stationary, z-score trading can become a slow-motion disaster.

## 4. OU half-life intuition

A mean-reverting spread can be approximated by an Ornstein-Uhlenbeck process:

$$
ds_t = \kappa(\theta-s_t)dt+\sigma dW_t
$$

A discrete approximation is:

$$
\Delta s_t = a + b s_{t-1} + \varepsilon_t
$$

If:

$$
b<0
$$

then the spread mean-reverts.

The half-life is approximately:

$$
HL = \frac{\log(2)}{-\log(1+b)}
$$

or, for small $b$:

$$
HL \approx \frac{\log(2)}{-b}
$$

Half-life helps choose rolling windows and holding-period assumptions.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import itertools
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

try:
    from statsmodels.tsa.stattools import adfuller
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

SCIPY_AVAILABLE, STATSMODELS_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class PairsConfig:
    n_days: int = 2_500
    n_assets: int = 10
    train_fraction: float = 0.65
    validation_fraction: float = 0.15
    annualisation: int = 252
    z_window: int = 60
    entry_z: float = 2.0
    exit_z: float = 0.25
    max_pairs_selected: int = 3
    cost_per_dollar_turnover: float = 0.00025
    seed: int = 42


config = PairsConfig()
config

## 6. Synthetic universe

We simulate a universe with:

1. one true cointegrated pair;
2. one weaker cointegrated pair;
3. one false pair that is correlated but not cointegrated;
4. several independent/noisy assets.

This allows the pipeline to show both success cases and traps.

In [ ]:
def simulate_asset_universe(config: PairsConfig) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(config.seed)
    n = config.n_days

    dates = pd.bdate_range("2015-01-01", periods=n)

    # Common market factor.
    market_ret = 0.0002 + 0.010 * rng.standard_t(df=8, size=n) * np.sqrt((8 - 2) / 8)
    market_log = np.cumsum(market_ret)

    # Sector factor.
    sector_ret = 0.0001 + 0.007 * rng.standard_normal(n)
    sector_log = np.cumsum(sector_ret)

    assets = {}

    # True strong cointegrated pair: A/B.
    x_A = 4.6 + market_log + 0.20 * sector_log + np.cumsum(0.002 * rng.standard_normal(n))

    spread_AB = np.zeros(n)
    for t in range(1, n):
        spread_AB[t] = 0.93 * spread_AB[t - 1] + 0.015 * rng.standard_normal()

    beta_AB = 1.08
    alpha_AB = 0.12
    x_B = alpha_AB + beta_AB * x_A + spread_AB

    assets["asset_A"] = np.exp(x_A)
    assets["asset_B"] = np.exp(x_B)

    # Weaker cointegrated pair: C/D with changing noise.
    x_C = 4.4 + 0.85 * market_log - 0.10 * sector_log + np.cumsum(0.003 * rng.standard_normal(n))

    spread_CD = np.zeros(n)
    for t in range(1, n):
        vol = 0.018 if t < int(0.7 * n) else 0.030
        spread_CD[t] = 0.96 * spread_CD[t - 1] + vol * rng.standard_normal()

    beta_CD = 0.88
    alpha_CD = -0.08
    x_D = alpha_CD + beta_CD * x_C + spread_CD

    assets["asset_C"] = np.exp(x_C)
    assets["asset_D"] = np.exp(x_D)

    # False pair: E/F share market/sector but no stationary spread.
    x_E = 4.2 + 0.95 * market_log + 0.50 * sector_log + np.cumsum(0.004 * rng.standard_normal(n))
    x_F = 4.1 + 0.97 * market_log + 0.55 * sector_log + np.cumsum(0.004 * rng.standard_normal(n))

    # Add independent slow drifts that break cointegration.
    x_E += np.linspace(0, 0.35, n)
    x_F += np.linspace(0, -0.25, n)

    assets["asset_E"] = np.exp(x_E)
    assets["asset_F"] = np.exp(x_F)

    # Additional noisy assets.
    for idx in range(6, config.n_assets):
        name = f"asset_{chr(ord('A') + idx)}"
        loading_m = rng.uniform(0.4, 1.2)
        loading_s = rng.uniform(-0.4, 0.6)
        idio = np.cumsum(rng.uniform(0.004, 0.012) * rng.standard_normal(n))
        drift = rng.uniform(-0.00005, 0.00025) * np.arange(n)
        x = 4.0 + loading_m * market_log + loading_s * sector_log + idio + drift
        assets[name] = np.exp(x)

    prices = pd.DataFrame(assets)
    prices.insert(0, "date", dates)

    truth = {
        "true_cointegrated_pairs": [("asset_A", "asset_B"), ("asset_C", "asset_D")],
        "false_correlated_pair": ("asset_E", "asset_F"),
        "true_beta_AB": beta_AB,
        "true_beta_CD": beta_CD
    }

    return prices, truth


prices, truth = simulate_asset_universe(config)

prices.head(), truth

In [ ]:
asset_cols = [c for c in prices.columns if c != "date"]
log_prices = prices.copy()
for col in asset_cols:
    log_prices[col] = np.log(log_prices[col])

returns = prices[["date"]].copy()
for col in asset_cols:
    returns[col] = np.log(prices[col]).diff()

returns = returns.dropna().reset_index(drop=True)

plt.figure(figsize=(12, 6))
for col in asset_cols:
    plt.plot(prices["date"], prices[col] / prices[col].iloc[0], label=col, alpha=0.75)
plt.title("Synthetic Asset Universe, Normalised Prices")
plt.xlabel("Date")
plt.ylabel("Normalised price")
plt.legend(ncol=2)
plt.show()

## 7. Correlation diagnostics

We compute return correlations.

This is useful but insufficient.

A highly correlated pair may still be a bad pairs-trading candidate if the price spread trends.

In [ ]:
return_corr = returns[asset_cols].corr()

plt.figure(figsize=(8, 7))
plt.imshow(return_corr.values, vmin=-1, vmax=1)
plt.colorbar(label="Return correlation")
plt.xticks(range(len(asset_cols)), asset_cols, rotation=45, ha="right")
plt.yticks(range(len(asset_cols)), asset_cols)
plt.title("Return Correlation Matrix")
plt.tight_layout()
plt.show()

return_corr

## 8. Chronological split

We use:

1. train: pair discovery and hedge-ratio estimation;
2. validation: parameter sanity and selection;
3. test: final out-of-sample backtest.

Pairs must be selected before the test set.

In [ ]:
n = len(log_prices)
train_end = int(config.train_fraction * n)
validation_end = int((config.train_fraction + config.validation_fraction) * n)

train_prices = log_prices.iloc[:train_end].copy()
val_prices = log_prices.iloc[train_end:validation_end].copy()
test_prices = log_prices.iloc[validation_end:].copy()

split_metadata = {
    "n_total": int(n),
    "n_train": int(len(train_prices)),
    "n_validation": int(len(val_prices)),
    "n_test": int(len(test_prices)),
    "train_start": str(train_prices["date"].iloc[0]),
    "train_end": str(train_prices["date"].iloc[-1]),
    "validation_start": str(val_prices["date"].iloc[0]),
    "validation_end": str(val_prices["date"].iloc[-1]),
    "test_start": str(test_prices["date"].iloc[0]),
    "test_end": str(test_prices["date"].iloc[-1])
}

pd.Series(split_metadata)

## 9. OLS hedge ratio

For a candidate pair $x,y$, estimate on train:

$$
y_t = \alpha + \beta x_t + \epsilon_t
$$

Then spread:

$$
s_t = y_t-\alpha-\beta x_t
$$

We evaluate spread stationarity and mean reversion on the training period.

In [ ]:
def ols_hedge_ratio(log_price_df: pd.DataFrame, x_col: str, y_col: str):
    x = log_price_df[x_col].to_numpy()
    y = log_price_df[y_col].to_numpy()

    X = np.column_stack([np.ones(len(x)), x])
    beta_hat = np.linalg.pinv(X.T @ X) @ X.T @ y

    alpha = float(beta_hat[0])
    beta = float(beta_hat[1])

    spread = y - alpha - beta * x

    return alpha, beta, spread


def compute_spread(log_price_df: pd.DataFrame, x_col: str, y_col: str, alpha: float, beta: float):
    return log_price_df[y_col].to_numpy() - alpha - beta * log_price_df[x_col].to_numpy()


alpha_ab, beta_ab, spread_ab_train = ols_hedge_ratio(train_prices, "asset_A", "asset_B")

pd.Series({
    "estimated_alpha_AB": alpha_ab,
    "estimated_beta_AB": beta_ab,
    "true_beta_AB": truth["true_beta_AB"]
})

## 10. Stationarity and mean-reversion diagnostics

For each spread, estimate:

$$
\Delta s_t = a + b s_{t-1} + \varepsilon_t
$$

A more negative $b$ suggests stronger mean reversion.

Approximate half-life:

$$
HL \approx \frac{\log(2)}{-b}
$$

We also compute an ADF p-value if `statsmodels` is available.

If not, we still report the mean-reversion slope and half-life proxy.

In [ ]:
def mean_reversion_diagnostics(spread):
    spread = np.asarray(spread, dtype=float)
    s_lag = spread[:-1]
    ds = np.diff(spread)

    X = np.column_stack([np.ones(len(s_lag)), s_lag])
    coeff = np.linalg.pinv(X.T @ X) @ X.T @ ds

    intercept = float(coeff[0])
    slope = float(coeff[1])

    fitted = X @ coeff
    residuals = ds - fitted

    ssr = float(residuals @ residuals)
    sst = float(((ds - ds.mean()) @ (ds - ds.mean())))
    r2 = 1 - ssr / sst if sst > 0 else np.nan

    if slope < 0 and (1 + slope) > 0:
        half_life = np.log(2) / (-np.log(1 + slope))
    elif slope < 0:
        half_life = np.log(2) / (-slope)
    else:
        half_life = np.inf

    if STATSMODELS_AVAILABLE:
        try:
            adf_result = adfuller(spread, maxlag=1, regression="c", autolag=None)
            adf_stat = float(adf_result[0])
            adf_pvalue = float(adf_result[1])
        except Exception:
            adf_stat = np.nan
            adf_pvalue = np.nan
    else:
        adf_stat = np.nan
        adf_pvalue = np.nan

    return {
        "mr_intercept": intercept,
        "mr_slope": slope,
        "mr_r2": r2,
        "half_life_days": float(half_life),
        "adf_stat": adf_stat,
        "adf_pvalue": adf_pvalue,
        "spread_std": float(np.std(spread, ddof=1))
    }


mean_reversion_diagnostics(spread_ab_train)

## 11. Pair screening

We screen all asset pairs using training data.

For each pair:

1. estimate hedge ratio;
2. compute spread;
3. compute mean-reversion diagnostics;
4. compute train spread z-score behaviour;
5. compute return correlation.

Selection should prefer:

- negative mean-reversion slope;
- reasonable half-life;
- low ADF p-value if available;
- not purely correlation-driven;
- reasonable spread volatility.

In [ ]:
def screen_pairs(log_price_train: pd.DataFrame, returns_train: pd.DataFrame, asset_cols: list[str]) -> pd.DataFrame:
    rows = []

    for x_col, y_col in itertools.combinations(asset_cols, 2):
        alpha, beta, spread = ols_hedge_ratio(log_price_train, x_col, y_col)
        diag = mean_reversion_diagnostics(spread)

        corr = returns_train[[x_col, y_col]].corr().iloc[0, 1]

        rows.append({
            "x": x_col,
            "y": y_col,
            "alpha": alpha,
            "beta": beta,
            "return_corr": corr,
            **diag
        })

    out = pd.DataFrame(rows)

    # Ranking score. Lower is better.
    half_life = out["half_life_days"].replace([np.inf, -np.inf], np.nan)
    out["half_life_penalty"] = (half_life - 20).abs()
    out["adf_rank_component"] = out["adf_pvalue"].fillna(0.50)

    out["selection_score"] = (
        0.50 * out["adf_rank_component"].rank(pct=True)
        + 0.25 * out["half_life_penalty"].rank(pct=True)
        + 0.25 * (-out["mr_slope"]).rank(pct=True, ascending=False)
    )

    out = out.sort_values("selection_score").reset_index(drop=True)

    return out


returns_train = returns[returns["date"].isin(train_prices["date"])].copy()

pair_screen = screen_pairs(train_prices, returns_train, asset_cols)

pair_screen.head(15)

## 12. Correlation trap demonstration

We compare:

1. true cointegrated pair `asset_A` / `asset_B`;
2. false correlated pair `asset_E` / `asset_F`.

The false pair can have high correlation but unstable spread.

In [ ]:
def pair_diagnostics_row(pair_screen, x, y):
    mask1 = (pair_screen["x"] == x) & (pair_screen["y"] == y)
    mask2 = (pair_screen["x"] == y) & (pair_screen["y"] == x)
    return pair_screen[mask1 | mask2]


pair_diagnostics_row(pair_screen, "asset_A", "asset_B")

In [ ]:
pair_diagnostics_row(pair_screen, "asset_E", "asset_F")

In [ ]:
def plot_pair_spread(log_price_df, x, y, alpha, beta, title):
    spread = compute_spread(log_price_df, x, y, alpha, beta)
    z = (spread - np.mean(spread)) / np.std(spread, ddof=1)

    plt.figure(figsize=(12, 5))
    plt.plot(log_price_df["date"], z)
    plt.axhline(2, linestyle="--")
    plt.axhline(-2, linestyle="--")
    plt.axhline(0, linestyle=":")
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Train-normalised spread z-score")
    plt.show()


# True pair.
row_ab = pair_diagnostics_row(pair_screen, "asset_A", "asset_B").iloc[0]
plot_pair_spread(
    train_prices,
    row_ab["x"],
    row_ab["y"],
    row_ab["alpha"],
    row_ab["beta"],
    "Training Spread: True Cointegrated Pair A/B"
)

# False pair.
row_ef = pair_diagnostics_row(pair_screen, "asset_E", "asset_F").iloc[0]
plot_pair_spread(
    train_prices,
    row_ef["x"],
    row_ef["y"],
    row_ef["alpha"],
    row_ef["beta"],
    "Training Spread: False Correlated Pair E/F"
)

## 13. Pair selection

We select a small number of pairs from training data only.

Selection criteria:

- reasonable mean reversion;
- half-life not infinite;
- ADF p-value below threshold if available;
- avoid absurd hedge ratios;
- rank by selection score.

In production, this would require stronger validation and multiple-testing control.

In [ ]:
def select_pairs(pair_screen: pd.DataFrame, config: PairsConfig) -> pd.DataFrame:
    candidates = pair_screen.copy()

    candidates = candidates[
        (candidates["mr_slope"] < 0)
        & (candidates["half_life_days"] > 1)
        & (candidates["half_life_days"] < 120)
        & (candidates["beta"].abs() < 5)
        & (candidates["spread_std"] > 1e-6)
    ].copy()

    if STATSMODELS_AVAILABLE:
        candidates = candidates[candidates["adf_pvalue"] < 0.10].copy()

    selected = candidates.head(config.max_pairs_selected).reset_index(drop=True)

    return selected


selected_pairs = select_pairs(pair_screen, config)

selected_pairs

## 14. Spread z-score without look-ahead

For test trading, we use hedge ratios estimated on training data.

Rolling spread mean/std are computed using historical spread values only.

At time $t$, the position is based on information available at $t$, and applied to next-period spread change.

In [ ]:
def compute_pair_spread_series(log_price_df, pair_row):
    return compute_spread(
        log_price_df,
        pair_row["x"],
        pair_row["y"],
        float(pair_row["alpha"]),
        float(pair_row["beta"])
    )


def rolling_zscore(series, window):
    s = pd.Series(series)
    mean = s.rolling(window, min_periods=max(20, window // 3)).mean()
    std = s.rolling(window, min_periods=max(20, window // 3)).std()
    z = (s - mean) / std.replace(0, np.nan)
    return z.to_numpy()


def generate_positions_from_z(z, entry=2.0, exit=0.25):
    position = np.zeros(len(z))
    current = 0.0

    for t in range(len(z)):
        z_t = z[t]

        if not np.isfinite(z_t):
            position[t] = current
            continue

        if current == 0:
            if z_t > entry:
                current = -1.0  # short spread
            elif z_t < -entry:
                current = 1.0   # long spread
        else:
            if abs(z_t) < exit:
                current = 0.0

        position[t] = current

    return position

## 15. Backtest a single pair

Spread PnL proxy:

$$
PnL_{t+1} = position_t \cdot (s_{t+1}-s_t)
$$

If position is long spread and spread rises, PnL is positive.

Transaction-cost proxy:

$$
cost_t = c \cdot |\Delta position_t| \cdot (1+|\beta|)
$$

This is a simplified log-spread PnL, not a full execution simulator.

In [ ]:
def backtest_pair(log_price_df, pair_row, config: PairsConfig, period_label: str):
    spread = compute_pair_spread_series(log_price_df, pair_row)
    z = rolling_zscore(spread, config.z_window)
    position = generate_positions_from_z(z, config.entry_z, config.exit_z)

    spread_change_next = np.r_[np.diff(spread), 0.0]

    gross_pnl = position * spread_change_next

    turnover = np.abs(np.diff(np.r_[0.0, position]))
    cost = config.cost_per_dollar_turnover * turnover * (1.0 + abs(float(pair_row["beta"])))
    net_pnl = gross_pnl - cost

    out = pd.DataFrame({
        "date": log_price_df["date"].to_numpy(),
        "x": pair_row["x"],
        "y": pair_row["y"],
        "period": period_label,
        "spread": spread,
        "zscore": z,
        "position": position,
        "spread_change_next": spread_change_next,
        "gross_pnl": gross_pnl,
        "turnover": turnover,
        "cost": cost,
        "net_pnl": net_pnl
    })

    out["cum_net_pnl"] = out["net_pnl"].cumsum()

    return out


# Demonstrate on first selected pair, or A/B if selection is empty.
if len(selected_pairs) > 0:
    demo_pair = selected_pairs.iloc[0]
else:
    demo_pair = row_ab

demo_train_bt = backtest_pair(train_prices, demo_pair, config, "train")
demo_val_bt = backtest_pair(val_prices, demo_pair, config, "validation")
demo_test_bt = backtest_pair(test_prices, demo_pair, config, "test")

demo_test_bt.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(demo_test_bt["date"], demo_test_bt["zscore"], label="z-score")
plt.axhline(config.entry_z, linestyle="--")
plt.axhline(-config.entry_z, linestyle="--")
plt.axhline(config.exit_z, linestyle=":")
plt.axhline(-config.exit_z, linestyle=":")
plt.title(f"Test Spread Z-score: {demo_pair['x']} / {demo_pair['y']}")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(demo_test_bt["date"], demo_test_bt["cum_net_pnl"])
plt.axhline(0, linestyle="--")
plt.title(f"Test Cumulative Net PnL Proxy: {demo_pair['x']} / {demo_pair['y']}")
plt.xlabel("Date")
plt.ylabel("Cumulative log-spread PnL proxy")
plt.show()

## 16. Performance metrics

We compute:

- total net PnL;
- annualised return proxy;
- annualised volatility proxy;
- Sharpe proxy;
- max drawdown;
- turnover;
- trade fraction;
- hit rate.

These are spread-PnL diagnostics, not audited portfolio returns.

In [ ]:
def performance_summary(bt: pd.DataFrame, config: PairsConfig):
    pnl = bt["net_pnl"].to_numpy()
    cum = np.cumsum(pnl)
    drawdown = cum - np.maximum.accumulate(cum)

    active = bt["position"].abs() > 0

    return {
        "x": bt["x"].iloc[0],
        "y": bt["y"].iloc[0],
        "period": bt["period"].iloc[0],
        "n_days": int(len(bt)),
        "total_net_pnl": float(np.sum(pnl)),
        "mean_daily_pnl": float(np.mean(pnl)),
        "annualised_pnl_proxy": float(np.mean(pnl) * config.annualisation),
        "annualised_vol_proxy": float(np.std(pnl, ddof=1) * np.sqrt(config.annualisation)),
        "sharpe_proxy": float(np.mean(pnl) / np.std(pnl, ddof=1) * np.sqrt(config.annualisation)) if np.std(pnl, ddof=1) > 0 else np.nan,
        "max_drawdown_proxy": float(np.min(drawdown)),
        "mean_turnover": float(bt["turnover"].mean()),
        "total_turnover": float(bt["turnover"].sum()),
        "active_fraction": float(active.mean()),
        "hit_rate_active": float((bt.loc[active, "net_pnl"] > 0).mean()) if active.any() else np.nan,
        "total_cost": float(bt["cost"].sum())
    }


demo_perf = pd.DataFrame([
    performance_summary(demo_train_bt, config),
    performance_summary(demo_val_bt, config),
    performance_summary(demo_test_bt, config)
])

demo_perf

## 17. Backtest selected pairs

We backtest all selected pairs on train, validation, and test.

The test set is the real judgement period.

In [ ]:
def backtest_selected_pairs(selected_pairs, train_prices, val_prices, test_prices, config):
    all_bt = []
    all_perf = []

    for _, pair_row in selected_pairs.iterrows():
        for period_label, df in [
            ("train", train_prices),
            ("validation", val_prices),
            ("test", test_prices)
        ]:
            bt = backtest_pair(df, pair_row, config, period_label)
            all_bt.append(bt)
            all_perf.append(performance_summary(bt, config))

    if len(all_bt) == 0:
        return pd.DataFrame(), pd.DataFrame()

    return pd.concat(all_bt, ignore_index=True), pd.DataFrame(all_perf)


all_pair_bt, all_pair_perf = backtest_selected_pairs(
    selected_pairs,
    train_prices,
    val_prices,
    test_prices,
    config
)

all_pair_perf

## 18. Equal-weight portfolio of pairs

A simple pairs portfolio averages daily PnL across selected pairs:

$$
PnL^{portfolio}_t = \frac{1}{N} \sum_{i=1}^{N}PnL^{pair_i}_t
$$

This ignores capital allocation, margin, borrow, and execution constraints.

It is a research diagnostic.

In [ ]:
def build_pair_portfolio(all_pair_bt: pd.DataFrame, config: PairsConfig):
    if all_pair_bt.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = all_pair_bt.copy()
    df["pair"] = df["x"] + "_" + df["y"]

    portfolio_rows = []

    for period, g in df.groupby("period"):
        pivot = g.pivot_table(index="date", columns="pair", values="net_pnl", aggfunc="sum").fillna(0.0)
        portfolio_pnl = pivot.mean(axis=1)

        out = pd.DataFrame({
            "date": pivot.index,
            "period": period,
            "portfolio_net_pnl": portfolio_pnl,
            "cum_portfolio_net_pnl": portfolio_pnl.cumsum()
        })

        portfolio_rows.append(out)

    portfolio = pd.concat(portfolio_rows, ignore_index=True)

    perf_rows = []
    for period, g in portfolio.groupby("period"):
        pnl = g["portfolio_net_pnl"].to_numpy()
        cum = np.cumsum(pnl)
        dd = cum - np.maximum.accumulate(cum)

        perf_rows.append({
            "period": period,
            "n_days": len(g),
            "total_net_pnl": float(np.sum(pnl)),
            "annualised_pnl_proxy": float(np.mean(pnl) * config.annualisation),
            "annualised_vol_proxy": float(np.std(pnl, ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(np.mean(pnl) / np.std(pnl, ddof=1) * np.sqrt(config.annualisation)) if np.std(pnl, ddof=1) > 0 else np.nan,
            "max_drawdown_proxy": float(np.min(dd))
        })

    return portfolio, pd.DataFrame(perf_rows)


portfolio_bt, portfolio_perf = build_pair_portfolio(all_pair_bt, config)

portfolio_perf

In [ ]:
if not portfolio_bt.empty:
    plt.figure(figsize=(12, 6))
    for period, group in portfolio_bt.groupby("period"):
        plt.plot(group["date"], group["cum_portfolio_net_pnl"], label=period)
    plt.axhline(0, linestyle="--")
    plt.title("Equal-Weight Selected Pairs Portfolio")
    plt.xlabel("Date")
    plt.ylabel("Cumulative net PnL proxy")
    plt.legend()
    plt.show()
else:
    print("No selected pairs available for portfolio plot.")

## 19. False pair backtest

Now we explicitly backtest the false correlated pair.

This demonstrates why correlation alone is dangerous.

Even if the pair looks related, the spread may drift.

In [ ]:
false_pair_row = row_ef.copy()

false_train_bt = backtest_pair(train_prices, false_pair_row, config, "train")
false_val_bt = backtest_pair(val_prices, false_pair_row, config, "validation")
false_test_bt = backtest_pair(test_prices, false_pair_row, config, "test")

false_perf = pd.DataFrame([
    performance_summary(false_train_bt, config),
    performance_summary(false_val_bt, config),
    performance_summary(false_test_bt, config)
])

false_perf

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(false_test_bt["date"], false_test_bt["zscore"], label="False pair z-score")
plt.axhline(config.entry_z, linestyle="--")
plt.axhline(-config.entry_z, linestyle="--")
plt.axhline(0, linestyle=":")
plt.title("False Correlated Pair Test Z-score")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(false_test_bt["date"], false_test_bt["cum_net_pnl"])
plt.axhline(0, linestyle="--")
plt.title("False Correlated Pair Test PnL")
plt.xlabel("Date")
plt.ylabel("Cumulative net PnL proxy")
plt.show()

## 20. Hedge-ratio stability

A pair can be cointegrated in one period and break later.

We estimate hedge ratio over rolling windows:

$$
\beta_t^{rolling}
$$

If $\beta$ drifts substantially, the pair may be unstable.

In [ ]:
def rolling_hedge_ratio(log_price_df, x_col, y_col, window=252, step=21):
    rows = []

    for start in range(0, len(log_price_df) - window + 1, step):
        sub = log_price_df.iloc[start:start + window]
        alpha, beta, spread = ols_hedge_ratio(sub, x_col, y_col)
        diag = mean_reversion_diagnostics(spread)

        rows.append({
            "start_date": sub["date"].iloc[0],
            "end_date": sub["date"].iloc[-1],
            "x": x_col,
            "y": y_col,
            "alpha": alpha,
            "beta": beta,
            "half_life_days": diag["half_life_days"],
            "mr_slope": diag["mr_slope"],
            "adf_pvalue": diag["adf_pvalue"]
        })

    return pd.DataFrame(rows)


rolling_beta_demo = rolling_hedge_ratio(log_prices, demo_pair["x"], demo_pair["y"], window=252, step=21)
rolling_beta_false = rolling_hedge_ratio(log_prices, false_pair_row["x"], false_pair_row["y"], window=252, step=21)

rolling_beta_demo.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(rolling_beta_demo["end_date"], rolling_beta_demo["beta"], marker="o", label=f"{demo_pair['x']}/{demo_pair['y']}")
plt.plot(rolling_beta_false["end_date"], rolling_beta_false["beta"], marker="x", label=f"{false_pair_row['x']}/{false_pair_row['y']}")
plt.title("Rolling Hedge Ratio Stability")
plt.xlabel("Window end")
plt.ylabel("Beta")
plt.legend()
plt.show()

## 21. Cost sensitivity

Pairs trading can have high turnover.

We vary transaction costs and check whether the selected-pair portfolio survives.

In [ ]:
def portfolio_cost_sensitivity(selected_pairs, train_prices, val_prices, test_prices, base_config, cost_grid):
    rows = []

    for cost in cost_grid:
        cfg = PairsConfig(**{**asdict(base_config), "cost_per_dollar_turnover": cost})
        bt, perf = backtest_selected_pairs(selected_pairs, train_prices, val_prices, test_prices, cfg)

        if perf.empty:
            continue

        portfolio, pperf = build_pair_portfolio(bt, cfg)

        for _, row in pperf.iterrows():
            rows.append({
                "cost_per_dollar_turnover": cost,
                **row.to_dict()
            })

    return pd.DataFrame(rows)


cost_grid = [0.0, 0.00010, 0.00025, 0.00050, 0.00100, 0.00200]

cost_sensitivity = portfolio_cost_sensitivity(
    selected_pairs,
    train_prices,
    val_prices,
    test_prices,
    config,
    cost_grid
)

cost_sensitivity.head()

In [ ]:
if not cost_sensitivity.empty:
    plt.figure(figsize=(10, 6))
    for period, group in cost_sensitivity.groupby("period"):
        plt.plot(group["cost_per_dollar_turnover"], group["sharpe_proxy"], marker="o", label=period)
    plt.axhline(0, linestyle="--")
    plt.title("Pairs Portfolio Cost Sensitivity")
    plt.xlabel("Cost per dollar turnover")
    plt.ylabel("Sharpe proxy")
    plt.legend()
    plt.show()
else:
    print("No cost sensitivity results.")

## 22. Parameter sensitivity

Entry and exit thresholds can be overfit.

We test several entry thresholds on the test period for the selected pairs.

If performance changes wildly, the strategy is fragile.

In [ ]:
def threshold_sensitivity(selected_pairs, train_prices, val_prices, test_prices, base_config, entry_grid):
    rows = []

    for entry in entry_grid:
        cfg = PairsConfig(**{**asdict(base_config), "entry_z": entry})
        bt, perf = backtest_selected_pairs(selected_pairs, train_prices, val_prices, test_prices, cfg)

        if bt.empty:
            continue

        portfolio, pperf = build_pair_portfolio(bt, cfg)
        test_perf = pperf[pperf["period"] == "test"]

        if len(test_perf):
            rows.append({
                "entry_z": entry,
                **test_perf.iloc[0].to_dict()
            })

    return pd.DataFrame(rows)


entry_grid = [1.0, 1.5, 2.0, 2.5, 3.0]

threshold_sensitivity_df = threshold_sensitivity(
    selected_pairs,
    train_prices,
    val_prices,
    test_prices,
    config,
    entry_grid
)

threshold_sensitivity_df

In [ ]:
if not threshold_sensitivity_df.empty:
    plt.figure(figsize=(10, 6))
    plt.plot(threshold_sensitivity_df["entry_z"], threshold_sensitivity_df["sharpe_proxy"], marker="o")
    plt.axhline(0, linestyle="--")
    plt.title("Test Performance Sensitivity to Entry Threshold")
    plt.xlabel("Entry z-score")
    plt.ylabel("Sharpe proxy")
    plt.show()

## 23. Practical checklist for pairs trading research

Before trusting a pairs strategy, check:

1. **Cointegration, not just correlation**  
   Does the spread mean-revert?

2. **Train/test discipline**  
   Are pair selection and hedge ratios estimated before the test period?

3. **Hedge-ratio stability**  
   Does $\beta$ drift?

4. **Half-life realism**  
   Is mean reversion fast enough to trade?

5. **Cost sensitivity**  
   Does edge survive turnover costs?

6. **Parameter sensitivity**  
   Does performance depend on one lucky threshold?

7. **Break risk**  
   What happens when the economic relationship changes?

8. **Borrow and shorting constraints**  
   Can both legs be traded?

9. **Dollar neutrality versus beta neutrality**  
   What is being neutralised?

10. **Portfolio construction**  
   Are pairs independent or sharing hidden factor risk?

11. **Execution**  
   Are fills realistic?

12. **Multiple testing**  
   Did you screen hundreds of pairs and keep the lucky few?

## 24. Saving outputs

The notebook saves:

1. synthetic prices;
2. returns;
3. pair screen;
4. selected pairs;
5. selected-pair backtests;
6. selected-pair performance;
7. portfolio backtest;
8. portfolio performance;
9. false-pair backtest;
10. rolling hedge-ratio diagnostics;
11. cost sensitivity;
12. threshold sensitivity;
13. manifest.

In [ ]:
output_dir = Path("data/processed/statistical_arbitrage_pairs_v1")
output_dir.mkdir(parents=True, exist_ok=True)

prices_path = output_dir / "synthetic_prices.csv"
returns_path = output_dir / "synthetic_returns.csv"
pair_screen_path = output_dir / "pair_screen.csv"
selected_pairs_path = output_dir / "selected_pairs.csv"
all_pair_bt_path = output_dir / "selected_pair_backtests.csv"
all_pair_perf_path = output_dir / "selected_pair_performance.csv"
portfolio_bt_path = output_dir / "portfolio_backtest.csv"
portfolio_perf_path = output_dir / "portfolio_performance.csv"
false_bt_path = output_dir / "false_pair_backtest.csv"
false_perf_path = output_dir / "false_pair_performance.csv"
rolling_beta_demo_path = output_dir / "rolling_beta_selected_pair.csv"
rolling_beta_false_path = output_dir / "rolling_beta_false_pair.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
threshold_sensitivity_path = output_dir / "threshold_sensitivity.csv"
manifest_path = output_dir / "manifest.json"

prices.to_csv(prices_path, index=False)
returns.to_csv(returns_path, index=False)
pair_screen.to_csv(pair_screen_path, index=False)
selected_pairs.to_csv(selected_pairs_path, index=False)
all_pair_bt.to_csv(all_pair_bt_path, index=False)
all_pair_perf.to_csv(all_pair_perf_path, index=False)
portfolio_bt.to_csv(portfolio_bt_path, index=False)
portfolio_perf.to_csv(portfolio_perf_path, index=False)
false_test_bt.to_csv(false_bt_path, index=False)
false_perf.to_csv(false_perf_path, index=False)
rolling_beta_demo.to_csv(rolling_beta_demo_path, index=False)
rolling_beta_false.to_csv(rolling_beta_false_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
threshold_sensitivity_df.to_csv(threshold_sensitivity_path, index=False)

manifest = {
    "dataset_name": "statistical_arbitrage_pairs_outputs",
    "schema_version": "statistical_arbitrage_pairs_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "truth": truth,
    "statsmodels_available": STATSMODELS_AVAILABLE,
    "scipy_available": SCIPY_AVAILABLE,
    "selected_pairs": selected_pairs[["x", "y", "beta", "half_life_days", "adf_pvalue", "selection_score"]].to_dict(orient="records"),
    "portfolio_perf": portfolio_perf.to_dict(orient="records"),
    "false_pair_perf": false_perf.to_dict(orient="records"),
    "notes": [
        "Synthetic universe contains true cointegrated pairs and a false correlated pair.",
        "Pair screening uses training data only.",
        "Hedge ratios are estimated on training data and held fixed for validation/test backtests.",
        "Backtest uses spread PnL proxy and simple turnover cost proxy.",
        "Correlation trap demonstration explicitly compares true pair and false correlated pair.",
        "This is a research diagnostic notebook, not a production execution simulator."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

prices_path, pair_screen_path, selected_pairs_path, portfolio_perf_path, manifest_path

## 25. Limitations

### 25.1 Synthetic data

The notebook uses synthetic prices.

Real pairs trading faces:

- delistings;
- borrow constraints;
- corporate actions;
- futures rolls;
- changing tick sizes;
- exchange holidays;
- liquidity gaps.

### 25.2 Simplified cost model

The cost proxy is not a full execution model.

Real costs include:

- bid/ask spread;
- slippage;
- market impact;
- borrow;
- funding;
- fees;
- margin.

### 25.3 Static hedge ratio

The hedge ratio is fixed after training.

Real systems may use rolling hedge ratios, Kalman filters, or dynamic beta models.

### 25.4 Multiple testing

Screening many pairs can produce false discoveries.

A production process needs robust holdout tests and false discovery controls.

### 25.5 Cointegration breaks

A pair can stop mean-reverting.

Break detection and stop-loss logic are crucial.

### 25.6 PnL proxy

The PnL is based on log-spread changes, not exact dollar-neutral portfolio accounting.

### 25.7 No portfolio risk model

Selected pairs may share market, sector, or liquidity risk.

## 26. Research frontier and extensions

Important extensions include:

1. **Kalman filter hedge ratios**  
   Dynamic beta estimation.

2. **Johansen cointegration**  
   Multivariate cointegration baskets.

3. **Sparse pairs selection**  
   Avoid data-mined pair discovery.

4. **Regime-aware pairs trading**  
   Combine HMM regimes with pair activation.

5. **Pairs portfolio optimisation**  
   Allocate capital across spreads based on risk and correlation.

6. **Break detection**  
   Detect when a pair relationship fails.

7. **Machine learning spread forecasts**  
   Use nonlinear features for spread reversion probability.

8. **Execution-aware pairs backtesting**  
   Add bid/ask, borrow, slippage, and market impact.

9. **Cross-contract futures stat arb**  
   Calendar spreads and related commodity contracts.

10. **Chinese futures pairs**  
   Explore sector pairs, main/deferred contracts, night sessions, price limits, and roll effects.

## 27. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_11_information_coefficient_analysis**  
   Evaluate pair signals as alpha factors.

2. **03_12_tree_models_for_return_prediction**  
   Nonlinear spread reversion classifiers.

3. **04_04_covariance_forecasting_dynamic_correlation**  
   Understand pair and portfolio risk.

4. **05_01_vectorized_backtest_engine**  
   Generalise this into reusable backtesting infrastructure.

5. **05_03_transaction_costs_slippage_latency**  
   Improve cost realism.

6. **07_01_multi_asset_cta_research_pipeline**  
   Integrate stat arb with broader futures research.

## 28. Summary

This notebook implemented statistical arbitrage pairs research.

It showed how to:

1. simulate a multi-asset universe with true and false pairs;
2. distinguish correlation from cointegration;
3. estimate hedge ratios with OLS;
4. compute spread stationarity and half-life diagnostics;
5. screen pairs using training data only;
6. generate z-score spread signals;
7. backtest selected pairs on train/validation/test;
8. create an equal-weight pair portfolio;
9. demonstrate a false correlated-pair trap;
10. test hedge-ratio stability;
11. test cost and threshold sensitivity;
12. save outputs and manifests.

The key computational takeaway:

> Pairs trading research is a full validation pipeline: pair discovery, hedge-ratio estimation, stationarity testing, signal construction, and cost-aware backtesting.

The key financial takeaway:

> A spread is only tradable if it mean-reverts fast enough, stays stable out of sample, and survives turnover costs.

## 29. Further reading

- Engle and Granger, "Co-integration and Error Correction."
- Gatev, Goetzmann, and Rouwenhorst, "Pairs Trading: Performance of a Relative-Value Arbitrage Rule."
- Vidyamurthy, *Pairs Trading.*
- Hamilton, *Time Series Analysis.*
- Literature on cointegration, Kalman-filter hedge ratios, statistical arbitrage, and spread trading.